In [17]:
!git clone https://github.com/gitdevqiang/SenWave.git

fatal: destination path 'SenWave' already exists and is not an empty directory.


In [18]:
import os
import pandas as pd
import numpy as np
import torch
#torchvision, torchaudio installed
from tqdm import tqdm

In [19]:
print(torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.cuda.empty_cache()

2.10.0
Using device: cpu


In [20]:
import warnings
import nltk
SEED = 0
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

torch.use_deterministic_algorithms(True)
warnings.filterwarnings('ignore')
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jessie_guo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
from datasets import load_dataset
import pandas as pd

df = pd.read_csv('SenWave/labeledtweets/labeledEn.csv')

lengths = [len(x) for x in df['Tweet']]
print(max(lengths))

df.head()
df.columns

140


Index(['ID', 'Tweet', 'Optimistic', 'Thankful', 'Empathetic', 'Pessimistic',
       'Anxious', 'Sad', 'Annoyed', 'Denial', 'Official report', 'Joking'],
      dtype='object')

In [22]:
label_cols = [
    "Optimistic",
    "Thankful",
    "Empathetic",
    "Pessimistic",
    "Anxious",
    "Sad",
    "Annoyed",
    "Denial",
    "Official report",
    "Joking"
]

text_col = "Tweet"
print(df[label_cols].sum())

df = df[[text_col] + label_cols].copy() #concatenate columns
print(df)
df = df.dropna(subset = [text_col]) #remove rows that have missing values in text_col column
print(df)
df[label_cols] = df[label_cols].fillna(0).astype(int) #selects label_cols columns, replaces all missing values with 0, convert to integer
df["labels"] = df[label_cols].values.tolist()
df = df.rename(columns = {text_col: "text"})
df = df[["text", "labels"]]
df.head()

classes = label_cols

Optimistic         2373
Thankful            498
Empathetic          389
Pessimistic        1325
Anxious            1695
Sad                2133
Annoyed            3492
Denial              631
Official report    1207
Joking             4476
dtype: int64
                                                  Tweet  Optimistic  Thankful  \
0     A glass of wine keeps the corona away- DRAKE. ...           1         0   
1     Can Anyone tell me if you took the flu shot la...           0         0   
2     Btw producers send me beats Im working on musi...           1         0   
3     When someone you know.. apart of your family d...           0         0   
4     Dear soccer, I really miss you ,please come ba...           0         0   
...                                                 ...         ...       ...   
9995  One good thing about the quarantine: I can now...           1         0   
9996  Shoutout to for making the coronavirus testing...           1         1   
9997  I find it am

In [23]:
from datasets import Dataset, DatasetDict

dataset = Dataset.from_pandas(df)
split = dataset.train_test_split(test_size = 0.2, seed = 0)
test_set = split["test"].train_test_split(test_size = 0.5, seed = 0)
senwave_dataset = DatasetDict({
    "train": split["train"],
    "validation": test_set["train"],
    "test": test_set["test"]
})
senwave_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1000
    })
})

In [30]:
from transformers import AutoTokenizer, DataCollatorWithPadding
PROBLEM_TYPE = "multi_label_classification"
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_labelling(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation = True,
        padding = "max_length",
        max_length = 128)
    return tokenized

if PROBLEM_TYPE == "single_label_classification":
    def onehot_to_class(example):
        example["labels"] = int(np.argmax(example["labels"]))
        return example

    single_label_emotions = senwave_dataset.map(onehot_to_class)
    print(single_label_emotions["train"][0]["labels"])

def problem_type(proc_dataset):
    tokenized_go_emotions = proc_dataset.map(tokenize_labelling, 
                                        batched = True, 
                                        remove_columns = ["text"])
    return tokenized_senwave

if PROBLEM_TYPE == "single_label_classification":
    tokenized_senwave = problem_type(single_label_emotions)
elif PROBLEM_TYPE == "multi_label_classification":
    tokenized_senwave = problem_type(senwave_dataset)
    
tokenized_senwave.set_format(
    type = "torch",
    columns = ["input_ids", "attention_mask", "labels"])

print(pd.DataFrame(tokenized_senwave['train']))
print(type(tokenized_senwave["train"][0]["labels"])) #type of labels for first example

class CustomDataCollator:
    def __call__(self, batch):
        input_ids = torch.stack([x["input_ids"] for x in batch])
        attention_mask = torch.stack([x["attention_mask"] for x in batch])
        labels = torch.stack([x["labels"] for x in batch]).float()  #.to(torch.float32)
        return {"input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels}

data_collator = CustomDataCollator()

SyntaxError: 'return' outside function (1428337196.py, line 17)

In [54]:
#Weights adjusted
y_train = np.array(senwave_dataset["train"]["labels"])
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
pos_weight = neg_counts/pos_counts
pos_weight = torch.tensor(pos_weight, dtype = torch.float)
print(pos_weight)

Map: 100%|█████████| 2000/2000 [00:00<00:00, 15275.90 examples/s]


                                                 labels  \
0     [tensor(1), tensor(0), tensor(0), tensor(0), t...   
1     [tensor(0), tensor(0), tensor(0), tensor(1), t...   
2     [tensor(0), tensor(0), tensor(0), tensor(0), t...   
3     [tensor(0), tensor(1), tensor(1), tensor(0), t...   
4     [tensor(0), tensor(0), tensor(0), tensor(0), t...   
...                                                 ...   
7995  [tensor(0), tensor(0), tensor(0), tensor(0), t...   
7996  [tensor(0), tensor(0), tensor(0), tensor(0), t...   
7997  [tensor(1), tensor(0), tensor(0), tensor(0), t...   
7998  [tensor(1), tensor(0), tensor(0), tensor(0), t...   
7999  [tensor(0), tensor(0), tensor(0), tensor(0), t...   

                                              input_ids  \
0     [tensor(101), tensor(2122), tensor(2420), tens...   
1     [tensor(101), tensor(2522), tensor(17258), ten...   
2     [tensor(101), tensor(2129), tensor(2515), tens...   
3     [tensor(101), tensor(2258), tensor(14876), ten...

In [ ]:
#Weights adjusted
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

class WeightedTrainer(Trainer): #creates new class inheriting from Trainer
    def __init__(self, *args, pos_weight=None, **kwargs): #*args passes any number of positional arguments, **kwargs pass any number of keyword (named) arguments.
        super().__init__(*args, **kwargs) #calls original Trainer
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs): #rewrites the fn in Trainer
        labels = inputs.pop("labels").float() #removes target answer from inputs dict, need to reserve for validation
        outputs = model(**inputs)
        logits = outputs.logits #raw scores model produces before turned into probabilities

        loss_fct = nn.BCEWithLogitsLoss( #stand. loss for multi-labelling
            pos_weight = self.pos_weight.to(logits.device) #if rare label missed, multiply penalty by pos_weight, .to ensures weights moved to same loc as model
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss #in case Trainer needs predictions for eval metrics

In [31]:
from sklearn.metrics import f1_score, precision_score, recall_score
#For multi-label classification
def find_best_threshold(logits, labels):
    probs = 1/(1 + np.exp(-logits))
    best_t_micro = 0.5
    best_t_macro = 0.5
    best_t_weighted = 0.5
    best_f1_micro = 0
    best_f1_macro = 0
    best_f1_weighted = 0

    for t in np.arange(0.1, 0.9, 0.05):
        preds = (probs > t).astype(int)
        f1_micro = f1_score(labels, preds, average = "micro", zero_division = 0)
        f1_macro = f1_score(labels, preds, average = "macro", zero_division = 0)
        f1_weighted = f1_score(labels, preds, average = "weighted", zero_division = 0)
        if f1_micro > best_f1_micro:
            best_f1_micro = f1_micro
            best_t_micro = t
        if f1_macro > best_f1_macro:
            best_f1_macro = f1_macro
            best_t_macro = t
        if f1_weighted > best_f1_weighted:
            best_f1_weighted = f1_weighted
            best_t_weighted = t

    return best_t_macro, best_t_micro, best_t_weighted, best_f1_macro, best_f1_micro, best_f1_weighted

In [44]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, precision_recall_fscore_support, hamming_loss
classes = label_cols
PROBLEM_TYPE = "multi_label_classification"

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if PROBLEM_TYPE == "multi_label_classification":

        best_t_macro, best_t_micro, best_t_weighted, best_f1_macro, best_f1_micro, best_f1_weighted = find_best_threshold(logits, labels)
    
        probs = 1 / (1 + np.exp(-logits)) #remove for multi-class classification
    
        predictions = (probs > best_t_weighted).astype(int)  # Get the predicted class labels

        hamming_loss_score = hamming_loss(labels, predictions)

    elif PROBLEM_TYPE == "single_label_classification":

        predictions = logits.argmax(axis=-1)  # Get the predicted class labels

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    f1_micro = f1_score(labels, predictions, average = "micro", zero_division = 0)
    f1_macro = f1_score(labels, predictions, average = "macro", zero_division = 0)
    precision_micro = precision_score(labels, predictions, average="micro", zero_division=0)
    recall_micro = recall_score(labels, predictions, average="micro", zero_division=0)
    precision_macro = precision_score(labels, predictions, average="macro", zero_division=0)
    recall_macro = recall_score(labels, predictions, average="macro", zero_division=0)

    metrics_dict = {
        'f1_weighted': f1_weighted,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_micro': f1_micro,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
    }

    if PROBLEM_TYPE == "multi_label_classification":
        metrics_dict.update({
            'hamming_loss_score': hamming_loss_score,
            'best_threshold': best_t_weighted
        })
    
    return metrics_dict

def training_model(TRAIN_OUTPUT, SAVE_OUPUT):
    model_1 = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, 
                                                                 num_labels = len(classes), #len(classes),  
                                                                 problem_type = PROBLEM_TYPE) #"single_label_classification" for multi-class classification #load the weights in saved dtype. Without, doubles memory usage if weights originally torch.bfloat16
    model_1.to(device)
    
    training_args = TrainingArguments(
        output_dir = TRAIN_OUTPUT,
        num_train_epochs = 15,
        per_device_train_batch_size = 32,
        gradient_accumulation_steps = 4, #accumulate gradients across several batches and update once as opposed to updating weights every batch, simulates training with large batch size w/o storing, saves memory but slower training
        #gradient_checkpointing=True, #recomputes intermediate activations during forward pass as opposed to storing - extra computation but saves memory
        fp16 = False, #bf16 for fast mixed precision training, fp16 false for mps
        learning_rate = 2e-5, #initial lr
        logging_steps = 100, #controls how frequently to update + return loss
        eval_strategy = "epoch", #when to evaluate a model during training
        save_strategy = "epoch", #when to save
        weight_decay = 0.01, #L2 regularisation, helps prevent model from overfitting by discouraging large weights - uses AdamW, independent of weight updates --> better training stability
        metric_for_best_model = 'eval_f1_weighted',
        greater_is_better = True,
        load_best_model_at_end = True,
        save_total_limit = 2
    )
    
    trainer_kwargs = {
        "model": model_1,
        "args": training_args,
        "train_dataset": tokenized_senwave["train"],
        "eval_dataset": tokenized_senwave["validation"],
        "processing_class": tokenizer,
        "compute_metrics": compute_metrics,
        "callbacks": [EarlyStoppingCallback(early_stopping_patience=3)],
        #pos_weight: pos_weight
    }
    
    if PROBLEM_TYPE == "multi_label_classification":
        trainer_kwargs["data_collator"] = data_collator

    trainer = Trainer(**trainer_kwargs)    
    trainer.train()
    
    trainer.save_model(SAVE_OUPUT)
    tokenizer.save_pretrained(SAVE_OUPUT)

    return trainer

trainer_multilabel = training_model("expdistilbert_finetuned-senwave_multilabel", "expdistilbert_finetuned-senwave-final_multilabel")


SyntaxError: 'return' outside function (2882623288.py, line 96)

In [ ]:
PROBLEM_TYPE = "single_label_classification"
trainer_multiclass = training_model("expdistilbert_finetuned-goemotions_multiclass", "expdistilbert_finetuned-goemotions-final_multiclass")

In [ ]:
#Training and Validation Loss, Weighted_F1 Curves
import matplotlib.pyplot as plt
import pandas as pd

def loss_curves_and_weighted_f1(trainer):
    history = trainer.state.log_history
    df = pd.DataFrame(history)
    
    train_df = df[df['loss'].notna()]
    eval_df = df[df['eval_loss'].notna()]
    weighted_f1_df = df[df['eval_f1_weighted'].notna()]
    
    plt.figure(figsize=(10, 6))
    
    plt.plot(train_df['epoch'], train_df['loss'], label='Training Loss')
    plt.plot(eval_df['epoch'], eval_df['eval_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"Training and Validation Loss {trainer[-10:]}")
    plt.clf()
    
    plt.plot(weighted_f1_df['epoch'], weighted_f1_df['eval_f1_weighted'])
    plt.xlabel('Epochs')
    plt.ylabel('Weighted F1 Score')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"Weighted F1 Score {trainer[-10:]}")

loss_curves_and_weighted_f1(trainer_multilabel)
loss_curves_and_weighted_f1(trainer_multiclass)


In [60]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback

SAVE_OUTPUT = "distilbert_finetuned-senwave-final"

def save_to_device(SAVE_OUTPUT):
    model_new = AutoModelForSequenceClassification.from_pretrained(SAVE_OUTPUT)
    tokenizer_new = AutoTokenizer.from_pretrained(SAVE_OUTPUT)
    return model_new, tokenizer_new

model_new, tokenizer_new = save_to_device(SAVE_OUTPUT)

model_new.to(device)
model_new.eval()

Loading weights: 100%|█████████| 104/104 [00:00<00:00, 9020.75it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [61]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.texts = dataframe['text']
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts.iloc[index])
        text = " ".join(text.split())
        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,  # Add special tokens
            max_length=self.max_len,
            padding='max_length',  # Pad to max_length
            return_token_type_ids=True,
            return_tensors='pt',  # Return PyTorch tensors
            truncation=True  # Truncate sequences longer than max_length
        )
        
        input_ids = inputs['input_ids'].squeeze(0)  # Remove the added batch dimension
        attention_mask = inputs['attention_mask'].squeeze(0)  # Remove the added batch dimension
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
        }

In [62]:
def test():
    model_new.eval()
    all_outputs = []
    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(device) #, dtype=torch.long)
            attention_mask = data['attention_mask'].to(device) #, dtype=torch.long)
            outputs = model_new(input_ids = input_ids, attention_mask = attention_mask)
            logits = outputs.logits

            if PROBLEM_TYPE == "multi_label_classification":
                probs = torch.sigmoid(logits)
                all_outputs.extend(probs.cpu().numpy().tolist())
            elif PROBLEM_TYPE == "single_label_classification":
                probs = torch.softmax(logits, dim = -1)
                predictions = torch.argmax(probs, dim = -1)
                all_outputs.extend(predictions.cpu().numpy().tolist())  
    
    return all_outputs

In [49]:
lengths = [len(tokenizer_new.encode(text)) for text in senwave_dataset['train']['text']]
print(max(lengths)) #max token count across all tweets
print("95th percentile:", np.percentile(lengths, 95)) #95% of samples contain 34 tokens or fewer 

MAX_LEN = 128

64
95th percentile: 34.0


In [63]:
#preliminary test evaluations
texts_path = 'Data/Georgian/Vazha_ThreePoems/'
sources = ['en', 'orig', 'gem', 'ggl', 'gpt']
poems_path = texts_path
poem_names = []

output_base = "results/poems/sentiment/"
os.makedirs(output_base, exist_ok=True)

ge_poems_directory = os.fsencode(poems_path + 'orig')
for file in sorted(os.listdir(ge_poems_directory)):
    file = file.decode()
    if file.endswith(".md"):
        poem_names.append(file)

print(poem_names)

['01_aluda_ketelauri.md', '02_host_and_guest.md', '03_the_snake_eater.md', '06_that_in_truth_is_not_manliness.md']


In [64]:
from torch.utils.data import DataLoader

PROBLEM_TYPE = "multi_label_classification"

for source in sources:
    output_dir = os.path.join(output_base, source)
    os.makedirs(output_dir, exist_ok = True)
    
    for poem in poem_names:
        input_file = os.path.join(poems_path, source, poem)
        
        poem_text = [line.strip() for line in open(input_file, "r")]
        poem_df = pd.DataFrame(poem_text, columns=['text'])
        test_dataset = CustomDataset(poem_df, tokenizer_new, MAX_LEN)
        test_params = {'batch_size': 1, 'shuffle': False, 'num_workers': 0}
        test_loader = DataLoader(test_dataset, **test_params)
        
        if PROBLEM_TYPE == "multi_label_classification":
            test_probs = np.array(test())
        
            test_outputs = (test_probs >= 0.3).astype(int)    
        
            for j, label in enumerate(classes):
                poem_df[label] = test_outputs[:, j]
        
        elif PROBLEM_TYPE == "single_label_classification":
            test_predictions = np.array(test())

            poem_df["predicted_label_id"] = test_predictions
    
            poem_df["predicted_label"] = [classes[i] for i in test_predictions]

        output_file = os.path.join(output_dir, poem)
        poem_df.to_csv(output_file)

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Georgian/Vazha_ThreePoems/gem/01_aluda_ketelauri.md'

In [65]:

t01 = pd.read_csv("results/poems/sentiment/en/06_that_in_truth_is_not_manliness.md")
t01_text = t01["text"]
print(t01_text)

if PROBLEM_TYPE == "multi_label_classification":

    manual_labels = [
        ["Annoyed"],
        ["Optimistic", "Annoyed"],
        ["Annoyed"],
        ["Annoyed", "Thankful", "Empathetic"],
        ["Annoyed", "Sad"],
        ["Empathetic", "Annoyed"],
        ["Annoyed"],
        ["Annoyed", "Sad"],
        ["Annoyed"],
        ["Empathetic"]
    ]
    
    t01["manual_label"] = manual_labels
    
    for label in classes:
        t01[f"manual_{label}"] = 0
    
    for i, labels_for_row in enumerate(manual_labels):
        for label in labels_for_row:
            if label in classes:
                t01.at[i, f"manual_{label}"] = 1
            else:
                print(f"Invalid label in row {i}: {label}")
    
    manual_cols = [f"manual_{label}" for label in classes]
    pred_cols = classes
    print(t01)

    y_true = t01[manual_cols].astype(int).values
    y_pred = t01[pred_cols].astype(int).values

elif PROBLEM_TYPE == "single_label_classification":
    t01["manual_label"] = [
    "Annoyed",
    "Optimistic",
    "Annoyed",
    "Thankful",
    "Annoyed",
    "Empathetic",
    "Annoyed",
    "Annoyed",
    "Annoyed",
    "Empathetic",
]

    label_to_id = {label: i  for i, label in enumerate(classes)}
    
    t01["manual_label_id"] = t01["manual_label"].map(label_to_id)
    print(t01)
    
    y_true = t01["manual_label_id"].values
    y_pred = t01["predicted_label_id"].values
    
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average = "micro", zero_division = 0)
f1_macro = f1_score(y_true, y_pred, average = "macro", zero_division = 0)
precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)

print(
    'f1_weighted:', f1_weighted,
    '\nprecision_weighted:', precision_weighted,
    '\nrecall_weighted:', recall_weighted,
    '\nf1_micro:', f1_micro,
    '\nprecision_micro:', precision_micro,
    '\nrecall_micro:', recall_micro,
    '\nf1_macro:', f1_macro,
    '\nprecision_macro:', precision_macro,
    '\nrecall_macro:', recall_macro,
)

0    That in truth is not manliness When someone eg...
1    A man would I call you only if Of your own inc...
2    Nor to call it manliness would I wish When you...
3    A man would I call you only when One throttled...
4    Tell me, who has styled it manly virtue When i...
5    Rather is it true manliness When you suffer fo...
6    And no manliness begins When with full stomach...
7    Or when you behave as superior to others, And ...
8    When from your minstrelsy you give none peace ...
9    A man then surely would I call you Were you to...
Name: text, dtype: object
   Unnamed: 0                                               text  Optimistic  \
0           0  That in truth is not manliness When someone eg...           0   
1           1  A man would I call you only if Of your own inc...           0   
2           2  Nor to call it manliness would I wish When you...           0   
3           3  A man would I call you only when One throttled...           0   
4           4  Tell me, 